# I-JEPA with YOLO26 on Football Images

This tutorial trains a YOLO26 backbone with an I-JEPA-style latent prediction objective. Training reads image pixels only. Football labels are used after training only to color the representation visualization.

**Learning goals**

- Understand context and target regions in I-JEPA
- Build the reusable I-JEPA module around YOLO26
- Train with AMP and two T4 GPUs
- Save a detector-compatible YOLO26 backbone
- Inspect training history and latent-space variation
- Visualize validation features with t-SNE


## How this I-JEPA adaptation learns

A target encoder produces the complete spatial feature map. Several rectangular target blocks are sampled on that map. Their corresponding image regions are hidden from the online context encoder. A lightweight transformer predicts the target latents from the visible context.

For predicted target features $p$ and stop-gradient target features $z$, the objective uses smooth L1 regression $L=SmoothL1(p,z)$. After each optimizer step, target parameters follow $xi=m xi+(1-m) theta$.

This is a compute-scaled YOLO-native I-JEPA adaptation, not a reproduction of the original ViT-H I-JEPA training regime.


## Roadmap

1. Configure Kaggle
2. Inspect the football images
3. Configure I-JEPA
4. Visualize target blocks
5. Dry-run the reusable module
6. Train on T4 x2
7. Review loss and checkpoints
8. Extract validation features
9. Plot t-SNE and nearest neighbors
10. Complete an exercise


## 1. Configure Kaggle

Select GPU T4 x2, enable Internet access, and attach the iasadpanwhar/football-player-detection-yolov8 dataset through Add Input.


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps "git+https://github.com/rifat963/ssl-detection-lab.git@main"


In [ ]:
from pathlib import Path
from collections import Counter
from dataclasses import replace
from importlib.metadata import version as installed_version
import json
import random

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
import yaml
from PIL import Image
from packaging.version import Version
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm
from ultralytics import YOLO

assert Version(installed_version("ssl-detection-lab")) >= Version("0.7.0")

import ssldet
from ssldet import PretrainConfig, launch_distributed_pretrain
from ssldet.backbones import YOLOBackboneEncoder
from ssldet.data import IMAGENET_MEAN, IMAGENET_STD, UnlabeledImageDataset, build_transform
from ssldet.ssl import create_ssl_module, sample_target_blocks

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
sns.set_theme(style="whitegrid", context="notebook")

assert torch.cuda.is_available(), "Select a GPU accelerator before continuing."
GPU_COUNT = torch.cuda.device_count()
DEVICE = torch.device("cuda:0")

pd.Series({
    "ssl-detection-lab": ssldet.__version__,
    "PyTorch": torch.__version__,
    "CUDA": torch.version.cuda,
    "GPU count": GPU_COUNT,
    "GPUs": ", ".join(torch.cuda.get_device_name(i) for i in range(GPU_COUNT)),
})


## 2. Inspect the dataset


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/iasadpanwhar/football-player-detection-yolov8/football_players_detection/football_players_detection"),
    Path("/kaggle/input/football-player-detection-yolov8/football_players_detection/football_players_detection"),
]
DATASET_ROOT = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if DATASET_ROOT is None:
    raise FileNotFoundError("Attach the football-player-detection-yolov8 dataset")

SPLITS = {
    split: {
        "images": DATASET_ROOT / split / "images",
        "labels": DATASET_ROOT / split / "labels",
    }
    for split in ("train", "valid", "test")
}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files(directory):
    return sorted(
        path for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

summary = []
for split, paths in SPLITS.items():
    summary.append({
        "split": split,
        "images": len(image_files(paths["images"])),
        "labels": len(list(paths["labels"].glob("*.txt"))),
    })
pd.DataFrame(summary).set_index("split")


In [ ]:
train_images = image_files(SPLITS["train"]["images"])
sample_paths = random.Random(SEED).sample(train_images, min(8, len(train_images)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.flat:
    axis.axis("off")
for axis, path in zip(axes.flat, sample_paths):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name, fontsize=9)
plt.tight_layout()
plt.show()


## 3. Configure I-JEPA

The fast preset uses 160-pixel images, at most 2,000 training images, five epochs, and a small predictor. Set FAST_RUN to False for the complete 224-pixel, 25-epoch experiment.


In [ ]:
FAST_RUN = True
OUTPUT_DIR = Path("/kaggle/working/ijepa_yolo26_football")

config = PretrainConfig(
    method="ijepa",
    image_roots=[str(SPLITS["train"]["images"])],
    output_dir=str(OUTPUT_DIR),
    yolo_model="yolo26n.yaml",
    epochs=5 if FAST_RUN else 25,
    batch_size=8 if FAST_RUN else 16,
    image_size=160 if FAST_RUN else 224,
    workers=2,
    max_images=2000 if FAST_RUN else None,
    seed=SEED,
    learning_rate=3e-4,
    min_learning_rate=3e-6,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_accum_steps=1 if FAST_RUN else 2,
    gradient_clip=5.0,
    amp=True,
    projection_dim=128 if FAST_RUN else 256,
    momentum=0.996,
    final_momentum=1.0,
    num_target_blocks=2 if FAST_RUN else 4,
    target_scale_min=0.10,
    target_scale_max=0.25,
    target_aspect_min=0.75,
    target_aspect_max=1.50,
    predictor_depth=1 if FAST_RUN else 2,
    predictor_heads=4,
    save_every=1,
).validate()

pd.Series({
    "preset": "fast" if FAST_RUN else "full",
    "epochs": config.epochs,
    "maximum images": config.max_images or len(train_images),
    "image size": config.image_size,
    "batch per GPU": config.batch_size,
    "target blocks": config.num_target_blocks,
    "projection dimensions": config.projection_dim,
    "AMP": config.amp,
})


## 4. Visualize an input and its latent target blocks


In [ ]:
preview_dataset = UnlabeledImageDataset(train_images, build_transform(config))
preview = preview_dataset[0]
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
display_image = (preview * std + mean).clamp(0, 1).permute(1, 2, 0)

grid_size = config.image_size // 32
masks = sample_target_blocks(
    1,
    grid_size,
    grid_size,
    config.num_target_blocks,
    (config.target_scale_min, config.target_scale_max),
    (config.target_aspect_min, config.target_aspect_max),
    torch.device("cpu"),
)[0]

fig, axes = plt.subplots(1, config.num_target_blocks + 1, figsize=(5 * (config.num_target_blocks + 1), 5))
axes[0].imshow(display_image)
axes[0].set_title("Augmented input")
for index, mask in enumerate(masks, start=1):
    axes[index].imshow(mask, cmap="Reds", vmin=0, vmax=1)
    axes[index].set_title(f"Latent target {index}")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()


The masks are sampled on the final YOLO backbone grid. During training, their union is expanded to image resolution and removed from the context input.


## 5. Dry-run the reusable I-JEPA module


In [ ]:
probe_detector = YOLO(config.yolo_model)
probe_encoder = YOLOBackboneEncoder(probe_detector.model).to(DEVICE)
feature_channels, _, _ = probe_encoder.infer_dimensions(config.image_size, DEVICE)
ijepa_module = create_ssl_module(
    "ijepa",
    encoder=probe_encoder,
    feature_channels=feature_channels,
    projection_dim=config.projection_dim,
    predictor_depth=config.predictor_depth,
    predictor_heads=config.predictor_heads,
    momentum=config.momentum,
    num_target_blocks=config.num_target_blocks,
    target_scale=(config.target_scale_min, config.target_scale_max),
    target_aspect=(config.target_aspect_min, config.target_aspect_max),
).to(DEVICE)

with torch.inference_mode(), torch.amp.autocast("cuda"):
    example_loss = ijepa_module(preview.unsqueeze(0).to(DEVICE))

pd.Series({
    "feature channels": feature_channels,
    "example loss": float(example_loss),
    "target encoder trainable": any(
        parameter.requires_grad for parameter in ijepa_module.target_encoder.parameters()
    ),
})


In [ ]:
del ijepa_module, probe_encoder, probe_detector, example_loss
torch.cuda.empty_cache()


## 6. Train on T4 x2

The launcher starts one process per GPU. The progress bar reports smooth L1 loss, learning rate, and EMA momentum.


In [ ]:
training_result = launch_distributed_pretrain(
    config,
    num_processes=GPU_COUNT,
    config_path=OUTPUT_DIR / "ijepa_config.yaml",
    check=True,
)
training_result


## 7. Review loss and checkpoints


In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text())
BEST_SSL = OUTPUT_DIR / "best_ssl.pt"
YOLO_CHECKPOINT = Path(manifest["outputs"]["yolo_checkpoint"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=history, x="epoch", y="loss", marker="o", ax=axes[0])
sns.lineplot(data=history, x="epoch", y="ema_momentum", marker="o", ax=axes[1])
axes[0].set_title("I-JEPA latent prediction loss")
axes[1].set_title("Target encoder momentum")
for axis in axes:
    axis.xaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

pd.Series({
    "best SSL checkpoint": str(BEST_SSL),
    "YOLO backbone checkpoint": str(YOLO_CHECKPOINT),
    "best loss": manifest["best_loss"],
    "unlabelled images": manifest["unlabeled_images"],
    "world size": manifest["world_size"],
})


The YOLO checkpoint contains the trained online backbone and is the simplest starting point for downstream detection. The SSL checkpoint retains online and target encoders, projectors, predictor, optimizer, scheduler, and history.


## 8. Extract validation features


In [ ]:
yaml_candidates = sorted(DATASET_ROOT.parent.rglob("data.yaml"))
dataset_yaml = yaml_candidates[0] if yaml_candidates else None
metadata = yaml.safe_load(dataset_yaml.read_text()) if dataset_yaml else {}
raw_names = metadata.get("names", {})
if isinstance(raw_names, list):
    CLASS_NAMES = {index: name for index, name in enumerate(raw_names)}
elif isinstance(raw_names, dict):
    CLASS_NAMES = {int(index): name for index, name in raw_names.items()}
else:
    CLASS_NAMES = {}

def object_classes(label_path):
    if not label_path.exists():
        return []
    return [
        int(float(line.split()[0]))
        for line in label_path.read_text().splitlines()
        if line.strip()
    ]

validation_images = image_files(SPLITS["valid"]["images"])
class_frequency = Counter(
    class_id
    for path in validation_images
    for class_id in object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt")
)
if not CLASS_NAMES:
    CLASS_NAMES = {class_id: f"class {class_id}" for class_id in class_frequency}

def image_label(path):
    values = set(object_classes(SPLITS["valid"]["labels"] / f"{path.stem}.txt"))
    return min(values, key=lambda value: class_frequency[value]) if values else -1


In [ ]:
selected_paths = validation_images
if len(selected_paths) > 500:
    selected_paths = sorted(random.Random(SEED).sample(selected_paths, 500))

feature_transform = v2.Compose([
    v2.Resize(config.image_size + 32, antialias=True),
    v2.CenterCrop(config.image_size),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FeatureDataset(Dataset):
    def __len__(self):
        return len(selected_paths)

    def __getitem__(self, index):
        path = selected_paths[index]
        with Image.open(path) as image:
            tensor = feature_transform(image.convert("RGB"))
        return tensor, image_label(path), path.name

loader = DataLoader(FeatureDataset(), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
encoder = YOLOBackboneEncoder(YOLO(str(YOLO_CHECKPOINT)).model).to(DEVICE).eval()
feature_batches = []
label_batches = []
file_names = []
with torch.inference_mode(), torch.amp.autocast("cuda"):
    for images, labels, names in tqdm(loader, desc="Extracting I-JEPA features"):
        features = encoder(images.to(DEVICE, non_blocking=True))
        feature_batches.append(F.normalize(features.float(), dim=1).cpu())
        label_batches.append(labels.numpy())
        file_names.extend(names)
feature_matrix = torch.cat(feature_batches)
label_ids = np.concatenate(label_batches)

pd.Series({
    "images": len(feature_matrix),
    "dimensions": feature_matrix.shape[1],
    "mean feature standard deviation": feature_matrix.std(dim=0).mean().item(),
})


## 9. Plot t-SNE


In [ ]:
perplexity = min(30.0, max(2.0, (len(feature_matrix) - 1) / 3))
coordinates = TSNE(
    n_components=2,
    perplexity=perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=1000,
    random_state=SEED,
).fit_transform(feature_matrix.numpy())
plot_frame = pd.DataFrame({
    "t-SNE 1": coordinates[:, 0],
    "t-SNE 2": coordinates[:, 1],
    "class": [CLASS_NAMES.get(int(value), "unlabelled") for value in label_ids],
})
fig, axis = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=plot_frame,
    x="t-SNE 1",
    y="t-SNE 2",
    hue="class",
    palette="tab10",
    s=65,
    alpha=0.82,
    ax=axis,
)
axis.set_title("t-SNE of I-JEPA YOLO26 features")
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Nearest neighbors


In [ ]:
QUERY_INDEX = 0
similarities = feature_matrix @ feature_matrix[QUERY_INDEX]
neighbors = torch.topk(similarities, k=min(6, len(feature_matrix))).indices.tolist()
neighbors = [value for value in neighbors if value != QUERY_INDEX][:5]
display_indices = [QUERY_INDEX] + neighbors
fig, axes = plt.subplots(1, len(display_indices), figsize=(4 * len(display_indices), 4))
for position, (axis, index) in enumerate(zip(axes, display_indices)):
    with Image.open(selected_paths[index]) as image:
        axis.imshow(image.convert("RGB"))
    title = "Query" if position == 0 else f"Similarity {similarities[index]:.3f}"
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()


## Exercise

Create a second run with four smaller target blocks. Keep the image subset, seed, optimizer, and epoch count unchanged. Compare loss, feature variation, t-SNE neighborhoods, and downstream detection performance.


In [ ]:
exercise_config = replace(
    config,
    num_target_blocks=4,
    target_scale_min=0.05,
    target_scale_max=0.15,
    output_dir="/kaggle/working/ijepa_yolo26_four_targets",
).validate()
pd.Series({
    "target blocks": exercise_config.num_target_blocks,
    "minimum scale": exercise_config.target_scale_min,
    "maximum scale": exercise_config.target_scale_max,
    "output": exercise_config.output_dir,
})


## Practical checks

- Reduce batch size from 8 to 4 if CUDA memory is exhausted.
- I-JEPA is heavier than SimCLR because it evaluates online and target encoders and runs a latent predictor.
- The target encoder must remain frozen to gradients and change only through EMA.
- A lower SSL loss does not guarantee higher detection mAP.
- Fine-tune the saved YOLO checkpoint with labels before evaluating detection quality.
